# Layer 1 — Descriptive Analytics: GSO Health Monitor

**Portfolio:** Mario Casanova — Analytics Engineering for Nutanix Support Operations  
**Objective:** Monitor the operational health of the Global Support Organization (GSO) through key performance indicators that answer: *"What is happening right now?"*

---

## Framework

This notebook implements a **GSO Health Monitor Dashboard** that tracks:

| KPI | Description | Business Value |
|---|---|---|
| **TTR P50 / P90** | Median and 90th percentile Time to Resolution | Identifies chronic vs. tail-risk delays |
| **SLA Compliance Rate** | % of tickets resolved within SLA threshold | Direct contractual obligation metric |
| **Backlog Aging** | Ticket age distribution by priority and product cohort | Prevents accumulation of stale P2/P3 tickets |
| **NPS Trend** | Net Promoter Score over time | Leading indicator of customer churn risk |
| **Migration Impact** | TTR delta for VMware-to-AHV migration tickets | Quantifies operational cost of platform transitions |

---

**Data Sovereignty Note:** All data used here is synthetically generated by `src/data_generator.py`, modeling realistic GSO operations including 100,000 support tickets across 3 years.

## 0. Setup and Data Loading

In [ ]:
import sys
import os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))  # allow importing from src/

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy import stats

# Consistent visual style
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams['figure.figsize'] = (14, 5)
plt.rcParams['figure.dpi'] = 120

print('Libraries loaded successfully.')

In [ ]:
# Generate or load the synthetic dataset
data_path = '../data/synthetic/support_tickets.csv'

if os.path.exists(data_path):
    df = pd.read_csv(data_path, parse_dates=['created_at', 'resolved_at'])
    print(f'Loaded from cache: {df.shape}')
else:
    from src.data_generator import generate_support_tickets
    df = generate_support_tickets()
    os.makedirs('../data/synthetic', exist_ok=True)
    df.to_csv(data_path, index=False)
    print(f'Generated and saved: {df.shape}')

# Derived columns
df['created_month'] = df['created_at'].dt.to_period('M')
df['created_quarter'] = df['created_at'].dt.to_period('Q')
df['created_year'] = df['created_at'].dt.year

df.head(3)

## 1. Dataset Overview

In [ ]:
print('=== GSO DATASET OVERVIEW ===')
print(f'Total tickets:        {len(df):>10,}')
print(f'Date range:           {df.created_at.min().date()} → {df.created_at.max().date()}')
print(f'Unique clusters:      {df.cluster_id.nunique():>10,}')
print(f'Unique engineers:     {df.assigned_engineer.nunique():>10,}')
print(f'Total escalations:    {df.escalated.sum():>10,} ({df.escalated.mean():.1%})')
print(f'SLA breaches:         {df.sla_breached.sum():>10,} ({df.sla_breached.mean():.1%})')
print(f'NPS respondents:      {df.nps_score.notna().sum():>10,} ({df.nps_score.notna().mean():.1%})')
print()
print('Priority distribution:')
print(df.priority.value_counts().sort_index().to_frame('count').assign(
    pct=lambda x: (x['count'] / len(df) * 100).round(1)
))

## 2. TTR Analysis — P50 and P90 by Priority

**Why P90?** The median (P50) tells us the typical experience, but P90 reveals the tail-risk cases — the 10% of tickets that take the longest. For enterprise SRE planning, P90 matters more than the median when one delayed P1 ticket can cost millions in customer SLA penalties.

In [ ]:
SLA_HOURS = {'P1': 0.5, 'P2': 2.0, 'P3': 4.0, 'P4': 8.0}

ttr_stats = df.groupby('priority')['ttr_hours'].agg(
    count='count',
    p50=lambda x: x.quantile(0.50),
    p90=lambda x: x.quantile(0.90),
    mean='mean',
    std='std'
).round(2)
ttr_stats['sla_threshold_h'] = ttr_stats.index.map(SLA_HOURS)
ttr_stats['p90_vs_sla_ratio'] = (ttr_stats['p90'] / ttr_stats['sla_threshold_h']).round(1)
print('TTR Statistics by Priority:')
print(ttr_stats.to_string())

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# ── Left: P50 vs P90 bar chart with SLA reference lines ──
ax = axes[0]
priorities = ['P1', 'P2', 'P3', 'P4']
x = np.arange(len(priorities))
width = 0.35

bars_p50 = ax.bar(x - width/2, ttr_stats.loc[priorities, 'p50'], width,
                  label='P50 (Median)', color='#4C72B0', alpha=0.85)
bars_p90 = ax.bar(x + width/2, ttr_stats.loc[priorities, 'p90'], width,
                  label='P90', color='#DD8452', alpha=0.85)

# SLA threshold lines
for i, priority in enumerate(priorities):
    sla = SLA_HOURS[priority]
    ax.hlines(sla, i - 0.4, i + 0.4, colors='red', linewidths=2, linestyles='--', alpha=0.7)

ax.set_xticks(x)
ax.set_xticklabels(priorities)
ax.set_xlabel('Priority')
ax.set_ylabel('Hours')
ax.set_title('TTR P50 vs P90 by Priority\n(red dashed = SLA threshold)', fontsize=12)
ax.legend()
ax.set_yscale('log')  # log scale because P1 SLA is 0.5h vs P4 96h

# ── Right: TTR distribution violin plot ──
ax2 = axes[1]
sample = df.sample(5000, random_state=42)  # sample for visualization performance
sns.violinplot(data=sample, x='priority', y='ttr_hours', order=priorities,
               inner='quartile', palette='muted', ax=ax2, log_scale=True)
ax2.set_title('TTR Distribution by Priority (log scale)', fontsize=12)
ax2.set_xlabel('Priority')
ax2.set_ylabel('TTR (hours, log scale)')

plt.tight_layout()
plt.suptitle('Section 2: Time-to-Resolution Analysis', y=1.02, fontsize=13, fontweight='bold')
plt.show()

## 3. SLA Compliance Rate — By Region and Priority

SLA compliance is a contractual commitment. Breaking it down by **region × priority** reveals where operational gaps exist, enabling targeted SRE staffing decisions.

In [ ]:
# SLA compliance heatmap
sla_compliance = df.groupby(['region', 'priority']).apply(
    lambda g: (1 - g['sla_breached'].mean()) * 100
).unstack('priority').round(1)

sla_compliance = sla_compliance[['P1', 'P2', 'P3', 'P4']]  # reorder columns

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(
    sla_compliance,
    annot=True, fmt='.1f', cmap='RdYlGn',
    vmin=50, vmax=100, linewidths=0.5,
    cbar_kws={'label': 'SLA Compliance %'},
    ax=ax
)
ax.set_title('SLA Compliance Rate (%) by Region × Priority\n(green = good, red = needs attention)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Priority Tier')
ax.set_ylabel('Region')
plt.tight_layout()
plt.show()

print('\nOverall SLA Compliance by Priority:')
overall_sla = (1 - df.groupby('priority')['sla_breached'].mean()) * 100
print(overall_sla.round(1).to_string())

## 4. Backlog Aging Analysis — By Product Version Cohort

Backlog aging is a **leading indicator** of SLA risk. Here we analyze the age distribution of open tickets segmented by AOS version — a proxy for product complexity and engineering debt.

In [ ]:
# Simulate 'open' tickets: those created in last 90 days of dataset
cutoff_date = df['created_at'].max() - pd.Timedelta(days=90)
open_tickets = df[df['created_at'] >= cutoff_date].copy()
open_tickets['age_hours'] = (df['created_at'].max() - open_tickets['created_at']).dt.total_seconds() / 3600

print(f'Simulated open tickets (last 90 days): {len(open_tickets):,}')

# Aging buckets
bins = [0, 4, 24, 72, 168, float('inf')]
labels = ['<4h', '4-24h', '1-3d', '3-7d', '>7d']
open_tickets['age_bucket'] = pd.cut(open_tickets['age_hours'], bins=bins, labels=labels)

# Pivot by AOS version cohort
aging_pivot = open_tickets.groupby(['aos_version', 'age_bucket']).size().unstack('age_bucket').fillna(0)

fig, ax = plt.subplots(figsize=(14, 6))
aging_pivot.plot(kind='bar', stacked=True, ax=ax,
                 colormap='RdYlGn_r', alpha=0.85, edgecolor='white', linewidth=0.3)
ax.set_title('Backlog Aging by AOS Version Cohort\n(older versions tend to accumulate stale tickets)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('AOS Version')
ax.set_ylabel('Number of Open Tickets')
ax.legend(title='Ticket Age', bbox_to_anchor=(1.01, 1), loc='upper left')
ax.tick_params(axis='x', rotation=30)
plt.tight_layout()
plt.show()

## 5. NPS Trend Analysis — Monthly with Confidence Bands

NPS is tracked monthly, with bootstrapped 95% confidence intervals to distinguish real trend signals from sampling noise.

In [ ]:
def bootstrap_ci(series, n_boot=500, ci=0.95):
    """Bootstrap confidence interval for mean NPS."""
    boots = [series.sample(len(series), replace=True).mean() for _ in range(n_boot)]
    lower = np.percentile(boots, (1 - ci) / 2 * 100)
    upper = np.percentile(boots, (1 + ci) / 2 * 100)
    return lower, upper

nps_df = df.dropna(subset=['nps_score']).copy()
nps_monthly = []

for period, group in nps_df.groupby('created_month'):
    mean_nps = group['nps_score'].mean()
    lo, hi = bootstrap_ci(group['nps_score'])
    nps_monthly.append({'period': period.to_timestamp(), 'nps_mean': mean_nps, 'ci_lo': lo, 'ci_hi': hi})

nps_trend = pd.DataFrame(nps_monthly)

fig, ax = plt.subplots(figsize=(16, 5))
ax.plot(nps_trend['period'], nps_trend['nps_mean'], color='#4C72B0', linewidth=2, label='Monthly Avg NPS')
ax.fill_between(nps_trend['period'], nps_trend['ci_lo'], nps_trend['ci_hi'],
                alpha=0.25, color='#4C72B0', label='95% CI (bootstrap)')

# Reference lines for NPS zones
ax.axhline(9, color='green', linestyle='--', alpha=0.5, linewidth=1, label='Promoter threshold (9)')
ax.axhline(6, color='orange', linestyle='--', alpha=0.5, linewidth=1, label='Detractor threshold (6)')

ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
plt.xticks(rotation=30)
ax.set_title('Monthly NPS Trend with 95% Bootstrap Confidence Intervals', fontsize=12, fontweight='bold')
ax.set_ylabel('NPS Score (0–10 scale)')
ax.set_xlabel('Month')
ax.legend(loc='lower left')
ax.set_ylim(0, 10)
plt.tight_layout()
plt.show()

## 6. Migration Impact Analysis

One of the most pressing business questions for the GSO during a major VMware-to-AHV transition is: **How much does active migration status affect TTR?** Here we test this rigorously using a Mann-Whitney U test (non-parametric, appropriate for skewed TTR distributions).

In [ ]:
migrating = df[df['is_migrating']]['ttr_hours']
not_migrating = df[~df['is_migrating']]['ttr_hours']

stat, p_value = stats.mannwhitneyu(migrating, not_migrating, alternative='greater')

print('=== MIGRATION IMPACT ON TTR ===')
print(f'Migrating tickets    — Median TTR: {migrating.median():.1f}h  | P90: {migrating.quantile(0.9):.1f}h  | N={len(migrating):,}')
print(f'Non-migrating tickets— Median TTR: {not_migrating.median():.1f}h  | P90: {not_migrating.quantile(0.9):.1f}h  | N={len(not_migrating):,}')
print(f'\nMann-Whitney U Test (migrating > non-migrating):')
print(f'  U statistic: {stat:.0f}')
print(f'  p-value:     {p_value:.4e}')
print(f'  Result:      {"✓ SIGNIFICANT" if p_value < 0.05 else "✗ NOT SIGNIFICANT"} at α=0.05')

# Effect size: median ratio
median_ratio = migrating.median() / not_migrating.median()
print(f'\nEffect size — Median TTR ratio (migrating / baseline): {median_ratio:.2f}x')

In [ ]:
# TTR by migration type and priority
pivot_migration = df.groupby(['migration_type', 'priority'])['ttr_hours'].median().unstack('priority')
pivot_migration = pivot_migration[['P1', 'P2', 'P3', 'P4']].round(1)

fig, ax = plt.subplots(figsize=(12, 5))
sns.heatmap(pivot_migration, annot=True, fmt='.1f', cmap='YlOrRd',
            linewidths=0.5, cbar_kws={'label': 'Median TTR (hours)'}, ax=ax)
ax.set_title('Median TTR (hours) by Migration Type × Priority\n(darker = longer resolution time)',
             fontsize=12, fontweight='bold')
ax.set_xlabel('Priority')
ax.set_ylabel('Migration Type')
plt.tight_layout()
plt.show()

## 7. Executive Summary

This cell generates a concise narrative suitable for a VP-level audience.

In [ ]:
# Compute key figures for the narrative
p1_sla_compliance = (1 - df[df['priority']=='P1']['sla_breached'].mean()) * 100
p1_p90 = df[df['priority']=='P1']['ttr_hours'].quantile(0.90)
overall_sla = (1 - df['sla_breached'].mean()) * 100
avg_nps = df['nps_score'].mean()
migration_ttr_impact = (migrating.median() / not_migrating.median() - 1) * 100
escalation_rate = df['escalated'].mean() * 100

summary = f"""
╔══════════════════════════════════════════════════════════════════════════╗
║          GSO OPERATIONAL HEALTH — EXECUTIVE SUMMARY                     ║
╠══════════════════════════════════════════════════════════════════════════╣
║  Period: {df.created_at.min().date()} → {df.created_at.max().date()}              ║
║  Total Tickets Analyzed: {len(df):,}                                  ║
║                                                                          ║
║  SLA PERFORMANCE                                                         ║
║  ├─ Overall Compliance:  {overall_sla:.1f}%                                 ║
║  ├─ P1 Compliance:       {p1_sla_compliance:.1f}%   (threshold: 30 min)          ║
║  └─ P1 P90 TTR:          {p1_p90:.1f}h   (target: <2h)                   ║
║                                                                          ║
║  CUSTOMER SATISFACTION                                                   ║
║  ├─ Average NPS Score:   {avg_nps:.2f}/10                                  ║
║  └─ Escalation Rate:     {escalation_rate:.1f}%                                  ║
║                                                                          ║
║  MIGRATION IMPACT                                                        ║
║  └─ VMware→AHV tickets take {migration_ttr_impact:.0f}% longer to resolve (median)   ║
║                                                                          ║
║  RECOMMENDATION                                                          ║
║  Prioritize SRE capacity increase in LATAM and APAC-INDIA during         ║
║  active VMware migration waves. Backlog aging in AOS 6.1.x cohorts      ║
║  requires targeted engineering review.                                   ║
╚══════════════════════════════════════════════════════════════════════════╝
"""
print(summary)

---

**Next Layer →** [Layer 2: Diagnostic Analytics — Anomaly Detection in Pulse Telemetry](./02_layer2_diagnostic_anomaly_detection.ipynb)

---
*Mario Casanova | Analytics Engineering Portfolio | Nutanix GSO*